In [0]:
from pyspark.sql import functions as F


df = spark.table("workspace.default.mall_customers_clean")
df.printSchema()
print("Rows:", df.count())
display(df.limit(5))

root
 |-- customer_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- annual_income_k: integer (nullable = true)
 |-- spending_score: integer (nullable = true)
 |-- age_group: string (nullable = true)
 |-- spending_category: string (nullable = true)
 |-- income_band: string (nullable = true)

Rows: 200


customer_id,gender,age,annual_income_k,spending_score,age_group,spending_category,income_band
1,Male,19,15,39,Teenager,Low Spender,Low Income
2,Male,21,15,81,Young Adult,High Spender,Low Income
3,Female,20,16,6,Teenager,Low Spender,Low Income
4,Female,23,16,77,Young Adult,High Spender,Low Income
5,Female,31,17,40,Young Adult,Medium Spender,Low Income


### Income and spending analysis

In [0]:
#  Overall spending score 
display(df.select(
    F.round(F.mean("spending_score"), 2).alias("mean"),
    F.round(F.stddev("spending_score"), 2).alias("std"),
    F.min("spending_score").alias("min"),
    F.max("spending_score").alias("max"),
    F.expr("percentile_approx(spending_score, 0.5)").alias("median")))

mean,std,min,max,median
50.2,25.82,1,99,50


In [0]:
# age group per spending 
cat_counts = df.groupBy("age_group").count() \
    .withColumn("pct", F.round(F.col("count") / df.count() * 100, 1))
display(cat_counts)


age_group,count,pct
Teenager,17,8.5
Young Adult,105,52.5
Senior,17,8.5
Middle Aged,61,30.5


In [0]:
#  By gender
display(df.groupBy("gender").agg(
    F.count("*").alias("customers"),
    F.round(F.avg("spending_score"), 2).alias("avg_spending_score"),
    F.round(F.avg("annual_income_k"), 2).alias("avg_income_k")))


gender,customers,avg_spending_score,avg_income_k
Male,88,48.51,62.23
Female,112,51.53,59.25


In [0]:
# By income band
display(df.groupBy("annual_income_k").agg(
    F.count("*").alias("customers"),
    F.round(F.avg("spending_score"), 2).alias("avg_spending_score")).orderBy("avg_spending_score", ascending=False))

annual_income_k,customers,avg_spending_score
69,2,74.5
99,2,68.0
23,2,63.5
38,2,63.5
76,2,63.5
15,2,60.0
29,2,59.0
97,2,59.0
79,2,59.0
17,2,58.0


In [0]:
# Spending category x gender / age group
df1 = (df
    .withColumn("spending_category",
        F.when(F.col("spending_score") >= 70, "High Spender")
         .when(F.col("spending_score") >= 40, "Medium Spender")
         .otherwise("Low Spender"))
    .withColumn("income_band",
        F.when(F.col("annual_income_k") < 40, "Low Income")
         .when(F.col("annual_income_k") < 70, "Medium Income")
         .otherwise("High Income"))
)
display(df1.groupBy("spending_category", "gender").count().orderBy("spending_category", "gender"))
display(df1.groupBy("age_group", "spending_category").count().orderBy("age_group", "spending_category"))

spending_category,gender,count
High Spender,Female,32
High Spender,Male,22
Low Spender,Female,30
Low Spender,Male,29
Medium Spender,Female,50
Medium Spender,Male,37


age_group,spending_category,count
Middle Aged,Low Spender,32
Middle Aged,Medium Spender,29
Senior,Low Spender,3
Senior,Medium Spender,14
Teenager,High Spender,2
Teenager,Low Spender,5
Teenager,Medium Spender,10
Young Adult,High Spender,52
Young Adult,Low Spender,19
Young Adult,Medium Spender,34


In [0]:
# Top and bottom 10 spenders
display(df.orderBy(F.desc("spending_score")).limit(10))
display(df.orderBy("spending_score").limit(10))

customer_id,gender,age,annual_income_k,spending_score,age_group
12,Female,35,19,99,Young Adult
20,Female,35,23,98,Young Adult
146,Male,28,77,97,Young Adult
186,Male,30,99,97,Young Adult
128,Male,40,71,95,Young Adult
168,Female,33,86,95,Young Adult
8,Female,23,18,94,Young Adult
164,Female,31,81,93,Young Adult
142,Male,32,75,93,Young Adult
42,Male,24,38,92,Young Adult


customer_id,gender,age,annual_income_k,spending_score,age_group
157,Male,37,78,1,Young Adult
159,Male,34,78,1,Young Adult
9,Male,64,19,3,Senior
33,Male,53,33,4,Middle Aged
31,Male,60,30,4,Middle Aged
135,Male,20,73,5,Teenager
23,Female,46,25,5,Middle Aged
163,Male,19,81,5,Teenager
141,Female,57,75,5,Middle Aged
7,Female,35,18,6,Young Adult


In [0]:
df1.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.mall_customers_clean")